# 🌊 Week 4, Day 3 — June 10, 2026
## Feature Engineering KPIs

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import seaborn as sns, os, json, warnings
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/Ocean_Plastic_Project"
DIRS = {"raw": BASE_DIR+"/data/raw", "processed": BASE_DIR+"/data/processed",
        "metadata": BASE_DIR+"/data/metadata", "visualizations": BASE_DIR+"/visualizations"}
LAT_MIN,LAT_MAX,LON_MIN,LON_MAX = 5,30,65,95
YEAR_MIN,YEAR_MAX = 2015,2026

In [ ]:
master  = pd.read_csv(DIRS['processed']+'/master_table_v1.csv')
df_temp = pd.read_csv(DIRS['processed']+'/temp_clean_v1.csv', parse_dates=['dt'])
df_co2  = pd.read_csv(DIRS['processed']+'/sea_level_co2_filtered.csv', parse_dates=['date'])
print(f'master: {master.shape}')
print(f'temp: {df_temp.shape}')
print(f'co2: {df_co2.shape}')

## KPI 1: Annual Temperature Change (°C/year)

In [ ]:
# Step 1: Compute annual average temperature from GlobalTemperatures
df_temp['year'] = df_temp['dt'].dt.year
annual_temp = (
    df_temp.groupby('year')['LandAndOceanAverageTemperature']
    .mean()
    .reset_index()
    .rename(columns={'LandAndOceanAverageTemperature': 'Avg_Temp_C'})
)

# Step 2: Temperature Change = current year avg - previous year avg
# This is a first-difference (delta) operation
annual_temp = annual_temp.sort_values('year').reset_index(drop=True)
annual_temp['Temp_Change_C'] = annual_temp['Avg_Temp_C'].diff().round(4)

print('Annual Temperature + Year-over-Year Change:')
print(annual_temp.to_string(index=False))
print()
print(f'NOTE: GlobalTemperatures only has data through 2015 in our filtered window.')
print(f'Temp_Change will be NaN for 2015 (first row — no prior year to diff against).')
print(f'For 2016 onwards we use the sea_level_co2 dataset which extends to 2026.')

In [ ]:
# Supplement with co2/ocean heat dataset for 2016–2026
# It doesn't have direct temperature but has ocean_heat_content_zj as proxy
df_co2['year'] = df_co2['date'].dt.year
co2_annual = (
    df_co2.groupby('year')[['sea_level_mm', 'ocean_heat_content_zj', 'co2_ppm']]
    .mean()
    .reset_index()
)

print('CO2/Sea-level dataset annual averages (environmental context):')
print(co2_annual.to_string(index=False))
print()
# Co2 year-over-year change
co2_annual['CO2_Change_ppm'] = co2_annual['co2_ppm'].diff().round(3)
co2_annual['Sea_Level_Change_mm'] = co2_annual['sea_level_mm'].diff().round(3)
print('CO2 and sea level year-over-year changes:')
print(co2_annual[['year','co2_ppm','CO2_Change_ppm','sea_level_mm','Sea_Level_Change_mm']].to_string(index=False))

In [ ]:
# Merge temperature change into Master Table by year
master_v2 = master.merge(annual_temp[['year','Avg_Temp_C','Temp_Change_C']], on='year', how='left')
master_v2 = master_v2.merge(co2_annual[['year','co2_ppm','CO2_Change_ppm','Sea_Level_Change_mm']], on='year', how='left')

print(f'Master Table after adding Temp_Change KPI: {master_v2.shape}')
print(f'New columns: Avg_Temp_C, Temp_Change_C, co2_ppm, CO2_Change_ppm, Sea_Level_Change_mm')
print()
print('Null check on new KPI columns:')
new_cols = ['Avg_Temp_C','Temp_Change_C','co2_ppm','CO2_Change_ppm','Sea_Level_Change_mm']
print(master_v2[new_cols].isnull().sum())

## KPI 2: Regional Contribution Ratio

In [ ]:
# Calculate each zone's share of total plastic density
# Only on rows that have plastic data (not species-only rows)
plastic_rows = master_v2.dropna(subset=['Plastic_Density_kg_km2'])
total_density = plastic_rows['Plastic_Density_kg_km2'].sum()

zone_totals = (
    plastic_rows.groupby('Ocean_Zone')['Plastic_Density_kg_km2']
    .sum()
    .reset_index()
    .rename(columns={'Plastic_Density_kg_km2': 'Zone_Total_Density'})
)
zone_totals['Regional_Contribution_Pct'] = (
    zone_totals['Zone_Total_Density'] / total_density * 100
).round(2)

print('Regional Contribution to Total Plastic Density:')
print(zone_totals.sort_values('Regional_Contribution_Pct', ascending=False).to_string(index=False))
print()
print(f'Total density (all zones): {total_density:.4f} kg/km²')

In [ ]:
# Merge Regional_Contribution_Pct back into master table per zone
master_v2 = master_v2.merge(
    zone_totals[['Ocean_Zone','Regional_Contribution_Pct']],
    on='Ocean_Zone', how='left'
)

print(f'Master Table with both KPIs: {master_v2.shape}')
print(f'Columns added: Temp_Change_C, Regional_Contribution_Pct')
print()
print('Sample rows with new KPI columns:')
kpi_cols = ['grid_lat','grid_lon','year','Ocean_Zone',
            'Plastic_Density_kg_km2','Temp_Change_C','Regional_Contribution_Pct']
print(master_v2[kpi_cols].dropna(subset=['Plastic_Density_kg_km2']).head(8).round(4).to_string(index=False))

## Visualize Both KPIs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Plot 1: Regional Contribution Bar ---
ax1 = axes[0]
colors = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6']
bars = ax1.barh(
    zone_totals.sort_values('Regional_Contribution_Pct')['Ocean_Zone'],
    zone_totals.sort_values('Regional_Contribution_Pct')['Regional_Contribution_Pct'],
    color=colors[:len(zone_totals)]
)
ax1.set_xlabel('Regional Contribution (%)')
ax1.set_title('KPI 2: Regional Contribution Ratio\nPlastic Density by Ocean Zone', fontweight='bold')
ax1.axvline(x=25, color='gray', linestyle='--', alpha=0.5, label='25% line')
for bar, val in zip(bars, zone_totals.sort_values('Regional_Contribution_Pct')['Regional_Contribution_Pct']):
    ax1.text(val+0.3, bar.get_y()+bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.legend()

# --- Plot 2: CO2 change as temp proxy ---
ax2 = axes[1]
ax2_twin = ax2.twinx()
valid_co2 = co2_annual.dropna(subset=['CO2_Change_ppm'])
ax2.bar(valid_co2['year'], valid_co2['CO2_Change_ppm'],
        color='#e74c3c', alpha=0.7, label='CO₂ Change (ppm/yr)')
ax2_twin.plot(co2_annual['year'], co2_annual['sea_level_mm'],
              color='steelblue', linewidth=2.5, marker='o', label='Sea Level (mm)')
ax2.set_xlabel('Year')
ax2.set_ylabel('CO₂ Change (ppm/year)', color='#e74c3c')
ax2_twin.set_ylabel('Sea Level (mm)', color='steelblue')
ax2.set_title('KPI 1: Environmental Change Indicators\n(CO₂ Δ + Sea Level Trend)', fontweight='bold')
ax2.legend(loc='upper left')
ax2_twin.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.suptitle('June 10, 2026 — Feature Engineering KPIs', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(DIRS['visualizations']+'/Week4_Day3_feature_kpis.png', dpi=150, bbox_inches='tight')
plt.show()
print('KPI visualization saved ✅')

In [ ]:
# Save updated master table
master_v2.to_csv(DIRS['processed']+'/master_table_v2.csv', index=False)
print(f'master_table_v2.csv saved — {master_v2.shape[0]:,} rows × {master_v2.shape[1]} cols ✅')
print(f'New columns vs v1: {set(master_v2.columns) - set(master.columns)}')